# 🧩 **CellPatcher**
___

CellPatcher turns the images of an experiment (and, for cell patches, their segmentation masks)
into reproducible **patch datasets**: small image crops plus a **patches.csv** table that records
where every patch came from. The patches are ready for feature extraction or deep-learning models.

Since version 0.2, CellPatcher reads every image and mask **through the run of the MorphoBricks
preprocessing notebook**: the preprocessing run has already checked the folders, the axes, the
masks and the intensities, so CellPatcher asks no axis, label or mask-pairing question. You point
it to the preprocessing run's output folder; that is all it needs to find your images.

### Patching strategies

| Strategy | What you get | Needs masks? |
|---|---|---|
| **Cell** | One patch per segmented object (cell, nucleus, ...) | Yes |
| **Grid** | Regular tiles covering every image | No |
| **Random** | Patches at random positions (reproducible with a seed) | Optional |
| **Whole FOV** | One patch per image | No |

### Quick workflow

1. Run the **Step 0** cell to start.
2. **Step 1:** paste the preprocessing output folder and press **Check the preprocessing run**.
3. **Step 2:** choose the patching strategy and the settings.
4. **Step 3:** press **Show the preview** to check the patches on one image.
5. **Step 4:** press **Patch and export every image**.

### Before you start

- [ ] You described your experiment and ran the **MorphoBricks preprocessing notebook** (version 0.5)
  on it, with your masks folder if you use masks, and it finished. Its output folder holds
  **morphobricks.toml**, **run_manifest.json** and **plane_histograms.csv**.
- [ ] Your images and masks have not changed since. If you added, removed or changed images, run
  the preprocessing notebook again first: CellPatcher refuses files the run did not record and
  never leaves an image out without telling you.

> 🛡️ **Your images are never changed.** The patches go to a separate output folder.

___
## 🧰 **Step 0: Start the notebook**

Click the cell below and press **Shift + Enter** (or the ▶ button in the toolbar). A green box
appears when everything is ready.

> **Colab users:** Step 0 connects your Google Drive (allow access in the window that opens)
> and installs CellPatcher **v0.5.0** from GitHub, which takes a minute or two. Keep the
> preprocessing output folder, your experiment and your results on Google Drive: in this
> notebook your Drive is the folder **/content/drive/MyDrive**. Everything else on the runtime is
> wiped when it disconnects; then run Step 0 again.
>
> **JupyterLab users:** Step 0 uses CellPatcher as installed for this notebook's kernel (or the
> copy of the repository around the notebook), installs what is missing, and prints the Python
> version and working folder. Nothing is mounted.

> 💡 **Tip:** the code of each cell is hidden; you only need to run the cells, from top to bottom,
> one after the other.

In [ ]:
# @title Run to start the CellPatcher notebook
import platform
import subprocess
import sys
from importlib import import_module, invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

# The notebook's version and name, as LabConstrictor's version check reads them
# (notebooks/notebook_latest_versions.yaml lists the latest version).
current_version = "0.5.0"
notebook_name = "CellPatcher"
# The release installed from GitHub when the package is missing (Google Colab, or a
# downloaded notebook). Set to the new tag, with current_version, before each release is
# tagged, so a tagged notebook installs its own release.
RELEASE_TAG = "4754dd1a02fd158378d87b4891f50e007a195dfa"
REPOSITORY = "https://github.com/CellMigrationLab/CellPatcher.git"
# The package's extras to install; on Colab also zarr, to read OME-Zarr images.
EXTRAS = "notebook"
# The notebook's interface in the package, as "module:class".
INTERFACE = "cellpatcher.notebook_ui:CellPatcherNotebook"

if "google.colab" in sys.modules:
    # Colab only: Google Drive holds the data and the results, which outlive the runtime.
    from google.colab import drive, output

    print(
        "Google Colab: connecting Google Drive. Allow access in the window that opens; "
        "your Drive is then the folder /content/drive/MyDrive.",
        flush=True,
    )
    try:
        drive.mount("/content/drive")
    except Exception as exc:  # a refused or interrupted authorisation
        raise RuntimeError(
            "Google Drive could not be connected. Run Step 0 again and allow access to "
            "your Drive; your data and output folders must be on Drive."
        ) from exc
    # Colab ships ipywidgets 7; this notebook's widgets need ipywidgets 8's front end.
    output.enable_custom_widget_manager()
    EXTRAS += ",zarr"
else:
    print(
        f"Local Jupyter: Python {platform.python_version()} on {platform.platform()}\n"
        f"Working folder: {Path.cwd()}",
        flush=True,
    )


def _missing_requirements() -> list[str]:
    """Return the notebook's requirements this kernel cannot import (or too old)."""
    required = (
        "cellpatcher",
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
        *(extra for extra in ("zarr", "torch") if extra in EXTRAS.split(",")),
    )
    missing = [name for name in required if find_spec(name) is None]
    try:
        widgets_major = int(version("ipywidgets").split(".")[0])
    except (PackageNotFoundError, ValueError):
        widgets_major = 8
    if widgets_major < 8 and "ipywidgets" not in missing:
        missing.append("ipywidgets 8")
    return missing


def _install_target() -> tuple[list[str], str]:
    """Return what pip installs when something is missing, and where CellPatcher comes from."""
    current = Path.cwd().resolve()
    checkout = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "setup.py").is_file() and (folder / "src/cellpatcher/__init__.py").is_file()
        ),
        None,
    )
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("cellpatcher")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel has already loaded another copy of CellPatcher. "
                "Restart the kernel, then run Step 0 to use this checkout."
            )
        # A local source tree works immediately, including in a git worktree.
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        return ["-e", f"{checkout}[{EXTRAS}]"], f"the local checkout at {checkout}"
    if find_spec("cellpatcher") is not None:
        try:
            return [f"cellpatcher[{EXTRAS}]"], f"the installed package, version {version('cellpatcher')}"
        except PackageNotFoundError:
            return [f"cellpatcher[{EXTRAS}]"], "the installed package"
    return [f"cellpatcher[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_TAG}"], f"release {RELEASE_TAG} from GitHub"


def _start_notebook() -> object:
    """Prepare this kernel's environment and return the notebook interface."""
    # This standalone cell can run before the package's Python requirement is checked.
    if sys.version_info < (3, 11):  # noqa: UP036
        raise RuntimeError("Select a Python 3.11 or newer notebook kernel, then run Step 0 again.")

    environment = "Google Colab" if "google.colab" in sys.modules else "local Jupyter"
    print(f"Checking the {environment} environment...", flush=True)
    install_target, origin = _install_target()
    missing = _missing_requirements()
    if missing:
        print(f"Installing notebook requirements ({', '.join(missing)})...", flush=True)
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *install_target],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output above, internet access and write permissions, then rerun Step 0. "
                "Installing from GitHub also needs Git."
            ) from exc
        invalidate_caches()
        missing = _missing_requirements()
        if missing:
            raise RuntimeError(
                f"Installation finished, but this kernel still cannot find {', '.join(missing)}. "
                "Restart the kernel and run Step 0 again."
            )
    loaded_widgets = str(getattr(sys.modules.get("ipywidgets"), "__version__", "8")).split(".")[0]
    if loaded_widgets.isdigit() and int(loaded_widgets) < 8:
        raise RuntimeError(
            "This kernel loaded ipywidgets 7 before ipywidgets 8 was installed. Restart the "
            "kernel (on Colab: Runtime > Restart session), then run Step 0 again."
        )

    print(f"Using {origin}.\nNotebook {notebook_name} {current_version}; Python: {sys.executable}", flush=True)
    module, _, name = INTERFACE.partition(":")
    try:
        interface = import_module(module)
    except ImportError as exc:
        raise RuntimeError(
            "CellPatcher was found, but the notebook interface could not be loaded. "
            "Read the original error below. If packages were changed in this session, "
            "restart the kernel and run Step 0 again."
        ) from exc
    return getattr(interface, name)()


notebook = _start_notebook()
notebook.welcome()

___
## 📂 **Step 1: Choose the preprocessing run**

> ⚠️ **Run preprocessing first.** CellPatcher does not preprocess: it needs the output folder of a
> finished, **described** run of the **MorphoBricks preprocessing notebook** (version 0.5): describe
> the experiment there (conditions, biological and technical repeats, channels, masks), publish the
> description, then run. A run without a published description is refused. If you have not done
> this on the experiment yet, do it first and come back here. (A later version will include the
> preprocessing steps in this notebook.)

Run the cell below, then:

1. Paste the path of the **preprocessing output folder** into **Preprocessing output folder**, or
   click **Browse…**. By default the preprocessing notebook writes it next to your experiment,
   named **<experiment>_MorphoBricks_Preprocessing**.
2. Only if you **moved or copied** the experiment (or its masks folder) since you preprocessed it,
   tick **The experiment was moved or copied since preprocessing** and give where they are now.
   Copy them with their file dates kept, or the run sees them as changed.
3. Press **Check the preprocessing run**. CellPatcher opens the run and checks that every image
   and mask is still as the run recorded it. A green card lists the images, conditions, channels
   and mask types. A red card says what to do, for example run preprocessing again after adding
   images.

> **Colab users:** paste paths that start with **/content/drive/MyDrive/**. **Browse…** cannot
> open a dialog on Colab: open the **Files** panel in the left bar, right-click a folder, choose
> **Copy path** and paste it. Use the experiment where you preprocessed it, on Drive.
>
> **JupyterLab users:** paste the path or use **Browse…**, which opens your computer's folder
> dialog.

In [ ]:
# @title Run to choose the preprocessing run
notebook.run_step()

___
## ⚙️ **Step 2: Choose the settings**

Run the cell below, then:

- **Patching strategy:** *Cell*, *Grid*, *Random* or *Whole FOV*. Only the settings of the chosen
  strategy are shown.
  - **Cell:** the **Mask type** (the run's types), the **Patch size** (per object, fixed, or from
    the objects of the whole dataset: press **Scan the object sizes of the dataset** first), the
    object area limits, border objects and whether to blank the pixels outside the object.
    A mask the run's description declares binary needs **The mask is binary: label its connected
    components (8-connected)**; without it the run is refused (`A_MASK_NOT_INSTANCES`).
  - **Grid:** the tile size, the overlap and the partial tiles at the edges.
  - **Random:** the patch size, the number of patches per image, the seed, and optionally
    sampling only inside the masks.
- **Planes** (images with several T frames): keep every frame in each patch, one patch per
  frame, or one frame. Images with more than one Z plane are refused (project or select first in
  morphobricks-preprocessing).
- **Processing:** a minimum mean intensity, a fixed output size, padding at the image edge, and
  which channels to split into single-channel patches.
- **Normalization** (optional): statistics from the **image** (recommended), the **biological
  repeat** (needs the experiment design confirmed in preprocessing) or each **patch**; the method;
  and whether files hold normalized or raw values. **Show parameters per group** lists the offset
  and scale of every group and channel (each patch becomes (pixels − offset) / scale), from the
  preprocessing run's statistics, before anything is patched; a group that cannot be normalized is
  shown in red.
- **Export:** TIFF or NumPy patch files.

Nothing is written until step 4.

In [ ]:
# @title Run to show the settings
notebook.settings_step()

___
## 🔍 **Step 3: Preview one image**

Run the cell below, choose an **Image** and press **Show the preview**. The image is patched with
the settings of step 2, in memory: rectangles show where the patches are, and the first patches are
shown below. With normalization, the first patches are also shown **before and after
normalization**, each with its intensity histogram. Nothing is written. Change the settings in
step 2 and press **Show the preview** again until the patches look right.

In [ ]:
# @title Run to show the preview
notebook.preview_step()

___
## 🚀 **Step 4: Patch and export**

Run the cell below. The **Output folder** is filled in next to the experiment; change it if you
like (it must be outside the experiment, masks and preprocessing folders). Then press
**Patch and export every image**. A progress bar counts the output folders.

The output folder holds:

```text
📂 my_experiment_CellPatcher
├── 📄 patches.csv        one row per patch: condition, repeats, image, coordinates, ...
├── 📄 images.csv         one row per image, also those without patches
├── 📄 run_config.json    every setting, and the preprocessing run it read
└── 📂 <condition>/<biological repeat>[/<technical repeat>]/   the patch files
```

> **Colab users:** keep the output folder on Google Drive (**/content/drive/MyDrive/...**), so
> the patches survive when the runtime disconnects.

In [ ]:
# @title Run to show the export
notebook.export_step()

___
## ✅ **After running**

- [ ] The output folder contains **patches.csv**, **images.csv** and **run_config.json**.
- [ ] Keep it together: the next MorphoBricks bricks read **patches.csv** and its
  **run_config.json**.

### Common problems

**"[unrecorded_files] ... that the preprocessing run did not record"**: images or masks were added
to the experiment after preprocessing. Run the preprocessing notebook again on the experiment as it
is (or exclude those images there), then check the run again in step 1.

**"[preprocessing_run_refused] ... changed or missing since the run"**: an image or mask changed,
or the experiment was moved. If you moved or copied it, tick **The experiment was moved or copied
since preprocessing** in step 1 and give where it is now; otherwise run preprocessing again.

**"[no_preprocessing_run]"** or a run written before v0.4: run the MorphoBricks preprocessing
notebook (version 0.5) on the experiment first.

**"[A_DESCRIPTION_REQUIRED]"**: the run has no published experiment description. Describe the
experiment in the preprocessing notebook, publish the description, run preprocessing again, then
check the run again in step 1.

**"[zarr_extra_missing]"**: the run holds OME-Zarr images. On Colab Step 0 installs what they
need; locally install CellPatcher with its zarr extra (`pip install "cellpatcher[zarr]"`).

**"... recorded N mask types ...; set matching.mask_type"**: choose the **Mask type** in step 2.

**"... the preprocessing run recorded no design"** with the biological-repeat scope: confirm the
experiment design in the preprocessing notebook (step 1 there), or use the image scope.

More help: the [documentation](https://github.com/CellMigrationLab/CellPatcher/blob/main/docs/README.md).